In [ ]:
from abc import ABC, abstractmethod
import torch
import torch.nn as nn
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

In [ ]:
class VelocityField(nn.Module, ABC):
    @abstractmethod
    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        pass

In [ ]:
class NeuralVelocityField(VelocityField):
    def __init__(self, d: int, hidden_dim: int = 128):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(d + 1, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, d)
        )

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        t = t.reshape(-1, 1).expand(x.shape[0], 1)

        return self.net(torch.cat([x, t], dim=-1))

In [ ]:
def train_velocity_field(
        model: NeuralVelocityField,
        data,
        steps=10000,
        lr: float = 1e-3
    ):
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    batch_size = 512

    for step in range(steps):
        idx = torch.randint(len(data), (batch_size,), device=data.device)
        x_1 = data[idx]
        x_0 = torch.randn_like(x_1)
        t = torch.rand(batch_size, 1, device=data.device)

        x_t = (1 - t) * x_0 + t * x_1
        target_v = x_1 - x_0

        pred_v = model(x_t, t)
        loss = (pred_v - target_v).square().sum(dim=-1).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 1000 == 0:
            print(f"Step {step}: FM Loss = {loss.item():.6f}")

In [ ]:
class EmpiricalVelocityField(VelocityField):
    def __init__(self, samples: torch.Tensor, learnable: bool = False) -> None:
        super().__init__()
        samples = samples.detach().clone()

        if learnable:
            self.samples = nn.Parameter(samples)
        else:
            self.register_buffer("samples", samples)

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        samples = self.samples

        sq_dist = torch.cdist(x, t * samples).square()
        weights = torch.softmax(-sq_dist / (2 * (1 - t)**2), dim=-1)

        return (weights @ samples - x) / (1 - t)

In [ ]:
class QuerySampler:
    def __init__(
        self,
        field: VelocityField,
        t_min: float = 0.35,
        t_max: float = 0.65,
        dt: float = 0.01,
        margin: float = 0.1,
    ):
        self.field = field
        self.t_min = t_min
        self.t_max = t_max
        self.dt = dt
        self.margin = margin

    @torch.no_grad()
    def integrate(self, x: torch.Tensor, t0: float, t1: float) -> torch.Tensor:
        steps = max(1, round((t1 - t0) / self.dt))
        dt = (t1 - t0) / steps

        for i in range(steps):
            t = x.new_tensor(t0 + i * dt)
            x = x + dt * self.field(x, t)

        return x

    @torch.no_grad()
    def sample(self, x0: torch.Tensor, n_queries: int = 512) -> list:
        # Collect trajectory endpoints
        endpoints = self.integrate(x0, 0.0, self.t_max)

        # Estimate bounding box
        low = endpoints.min(dim=0).values
        high = endpoints.max(dim=0).values

        padding = self.margin * (high - low)
        low = low - padding
        high = high + padding

        # Uniform sampling inside bounding box
        queries = []
        steps = max(1, round((self.t_max - self.t_min) / self.dt))

        for t in torch.linspace(
            self.t_min, self.t_max, steps + 1, device=x0.device
        ):
            x = low + (high - low) * torch.rand(
                n_queries, x0.shape[1], device=x0.device
            )
            queries.append((x, t))

        return queries

In [ ]:
def reconstruct(
        true_field: VelocityField,
        recon_field: EmpiricalVelocityField,
        queries: list,
        steps: int = 5000,
        batch_size: int = 128,
        lr: float = 0.05,
    ):
    optimizer = torch.optim.Adam(recon_field.parameters(), lr)

    for step in range(steps):
        x_all, t = queries[torch.randint(len(queries), (1,)).item()]
        idx = torch.randint(len(x_all), (batch_size,), device=x_all.device)
        x = x_all[idx]

        with torch.no_grad():
            target_v = true_field(x, t)

        pred_v = recon_field(x, t)
        loss = (pred_v - target_v).square().mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if step % 1000 == 0:
            print(f"Step {step}: Velocity MSE = {loss.item():.6f}")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# -------------------- Settings -------------------- #
torch.manual_seed(0)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

n = 100
d = 32
std = 0.3
scale = 3.0

# -------------------- Dataset -------------------- #
means = torch.zeros(4, d, device=device)
means[:, :2] = scale * torch.tensor([
    [-1, -1],
    [-1,  1],
    [ 1, -1],
    [ 1,  1],
], device=device)

labels = torch.arange(n, device=device) % 4
samples = means[labels] + std * torch.randn(n, d, device=device)

# -------------------- Velocity Fields -------------------- #
model = NeuralVelocityField(d, hidden_dim=256).to(device)
train_velocity_field(model, samples, steps=30000)
model.eval()

initial_samples = torch.randn_like(samples)
recon_field = EmpiricalVelocityField(
    initial_samples, learnable=True
).to(device)

In [ ]:
# -------------------- Query Sampling -------------------- #
sampler = QuerySampler(
    field=model,
    t_min=0.35,
    t_max=0.65,
    dt=0.01,
    margin=0.1,
)

x0 = torch.randn(8192, d, device=device)
queries = sampler.sample(x0, n_queries=1024)

print(f"Query Timesteps: {len(queries)}")
print(f"Queries per Timestep: {len(queries[0][0])}")

# -------------------- Reconstruction -------------------- #
reconstruct(
    true_field=model,
    recon_field=recon_field,
    queries=queries,
    steps=25000,
    batch_size=512,
    lr=0.01,
)

In [ ]:
true = samples.detach().cpu()
recon = recon_field.samples.detach().cpu()

# -------------------- Reconstruction Error -------------------- #
dist = torch.cdist(true, recon)
row, col = linear_sum_assignment(dist.numpy())

errors = dist[row, col]
d = true.shape[1]

print("Mean Distance:           ", errors.mean().item())
print("Normalized Mean Distance:", (errors.mean() / d**0.5).item())
print("Max Distance:            ", errors.max().item())
print("Median Distance:         ", errors.median().item())

print("Distance Quantiles:", torch.quantile(
    errors, torch.tensor([0.1, 0.25, 0.5, 0.75, 0.9])
).tolist())

# -------------------- Train vs. Test -------------------- #
test = (means[labels] + std * torch.randn_like(samples)).cpu()

train_dist = torch.cdist(true, recon).min(dim=1).values
test_dist = torch.cdist(test, recon).min(dim=1).values

print("\nTrain NN Distance:", train_dist.mean().item())
print("Test NN Distance: ", test_dist.mean().item())

# -------------------- PCA Visualization -------------------- #
combined = torch.cat([true, recon], dim=0)

pca = PCA(n_components=2)
projected = pca.fit_transform(combined.numpy())

true_2d = projected[:len(true)]
recon_2d = projected[len(true):]

plt.figure(figsize=(6, 6))
plt.scatter(true_2d[:, 0], true_2d[:, 1],
            label="Ground Truth", alpha=0.7)
plt.scatter(recon_2d[:, 0], recon_2d[:, 1],
            marker="x", label="Reconstructed", alpha=0.7)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.axis("equal")
plt.legend()
plt.show()